In [0]:
from pyspark.sql import functions as F

# 1. Capture your Silver updates (assume you've joined orders with their latest status)
# This example logic identifies the latest timestamp for each order stage
silver_updates = spark.table("workspace.amazon_fullfilment_silver.orders_silver") \
    .filter("is_current = true") \
    .groupBy("order_id", "customer_id") \
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )

# 2. Register as a temporary view to use SQL Merge
silver_updates.createOrReplaceTempView("updates_view")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    -- Calculate durations on the fly if timestamps exist
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(target.date_picking)) / 60 AS INT)
WHEN NOT MATCHED THEN
  INSERT (order_id, customer_id, date_initial, date_picking, date_shipped, current_status)
  VALUES (source.order_id, source.customer_id, source.initial_ts, source.picking_ts, source.shipped_ts, source.latest_status)
""")

In [0]:
from pyspark.sql import functions as F

silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true")
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )
    # 1. Populate date_key (YYYYMMDD) from the initial timestamp
    .withColumn("date_key", F.date_format("initial_ts", "yyyyMMdd").cast("int"))
    
    # 2. Determine is_completed based on Shipped status
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    # 3. Calculate total_fulfillment_time (Initial to Shipped) in minutes
    # We use unix_timestamp to get seconds, then divide by 60
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

# Register the updated logic as the view
silver_updates.createOrReplaceTempView("updates_view")

In [0]:
%sql
select * from updates_view

In [0]:
# catching up with data with Initial status
from pyspark.sql import functions as F

silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("status='Initial'")
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )
    # 1. Populate date_key (YYYYMMDD) from the initial timestamp
    .withColumn("date_key", F.date_format("initial_ts", "yyyyMMdd").cast("int"))
    
    # 2. Determine is_completed based on Shipped status
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    # 3. Calculate total_fulfillment_time (Initial to Shipped) in minutes
    # We use unix_timestamp to get seconds, then divide by 60
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

# Register the updated logic as the view
silver_updates.createOrReplaceTempView("updates_view1")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view1 AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    target.is_completed = source.is_completed,
    target.total_fulfillment_time = source.total_fulfillment_time,
    -- Calculate internal milestone durations
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(source.picking_ts)) / 60 AS INT),
    target._last_updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, date_key, 
    date_initial, date_picking, date_shipped, 
    total_fulfillment_time, current_status, is_completed, _last_updated_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.date_key, 
    source.initial_ts, source.picking_ts, source.shipped_ts, 
    source.total_fulfillment_time, source.latest_status, source.is_completed, current_timestamp()
  )""")

In [0]:
# catching up with data with none Initial status and expired ones
from pyspark.sql import functions as F

silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = false and status<>'Initial'")
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )
    # 1. Populate date_key (YYYYMMDD) from the initial timestamp
    .withColumn("date_key", F.date_format("initial_ts", "yyyyMMdd").cast("int"))
    
    # 2. Determine is_completed based on Shipped status
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    # 3. Calculate total_fulfillment_time (Initial to Shipped) in minutes
    # We use unix_timestamp to get seconds, then divide by 60
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

# Register the updated logic as the view
silver_updates.createOrReplaceTempView("updates_view1")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view1 AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    target.is_completed = source.is_completed,
    target.total_fulfillment_time = source.total_fulfillment_time,
    -- Calculate internal milestone durations
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(source.picking_ts)) / 60 AS INT),
    target._last_updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, date_key, 
    date_initial, date_picking, date_shipped, 
    total_fulfillment_time, current_status, is_completed, _last_updated_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.date_key, 
    source.initial_ts, source.picking_ts, source.shipped_ts, 
    source.total_fulfillment_time, source.latest_status, source.is_completed, current_timestamp()
  )""")

In [0]:
# main script
from pyspark.sql import functions as F

silver_updates = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .filter("is_current = true")
    .groupBy("order_id", "customer_id")
    .agg(
        F.min(F.when(F.col("status") == 'Initial', F.col("updated_timestamp"))).alias("initial_ts"),
        F.min(F.when(F.col("status") == 'Picking', F.col("updated_timestamp"))).alias("picking_ts"),
        F.min(F.when(F.col("status") == 'Shipped', F.col("updated_timestamp"))).alias("shipped_ts"),
        F.last("status").alias("latest_status")
    )
    # 1. Populate date_key (YYYYMMDD) from the initial timestamp
    .withColumn("date_key", F.date_format("initial_ts", "yyyyMMdd").cast("int"))
    
    # 2. Determine is_completed based on Shipped status
    .withColumn("is_completed", F.when(F.col("latest_status") == 'Shipped', True).otherwise(False))
    
    # 3. Calculate total_fulfillment_time (Initial to Shipped) in minutes
    # We use unix_timestamp to get seconds, then divide by 60
    .withColumn("total_fulfillment_time", 
        F.when(F.col("shipped_ts").isNotNull(), 
            (F.unix_timestamp("shipped_ts") - F.unix_timestamp("initial_ts")) / 60
        ).otherwise(None).cast("int")
    )
)

# Register the updated logic as the view
silver_updates.createOrReplaceTempView("updates_view1")

# 3. Perform the CUMULATIVE MERGE
spark.sql("""
MERGE INTO workspace.amazon_fullfilment_gold.fact_order_lifecycle AS target
USING updates_view1 AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.date_picking = COALESCE(target.date_picking, source.picking_ts),
    target.date_shipped = COALESCE(target.date_shipped, source.shipped_ts),
    target.current_status = source.latest_status,
    target.is_completed = source.is_completed,
    target.total_fulfillment_time = source.total_fulfillment_time,
    -- Calculate internal milestone durations
    target.minutes_to_pick = CAST((unix_timestamp(source.picking_ts) - unix_timestamp(target.date_initial)) / 60 AS INT),
    target.minutes_to_ship = CAST((unix_timestamp(source.shipped_ts) - unix_timestamp(source.picking_ts)) / 60 AS INT),
    target._last_updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, date_key, 
    date_initial, date_picking, date_shipped, 
    total_fulfillment_time, current_status, is_completed, _last_updated_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.date_key, 
    source.initial_ts, source.picking_ts, source.shipped_ts, 
    source.total_fulfillment_time, source.latest_status, source.is_completed, current_timestamp()
  )""")

In [0]:
%sql
select * from workspace.amazon_fullfilment_gold.fact_order_lifecycle

In [0]:
%sql
describe workspace.amazon_fullfilment_silver.address_silver

In [0]:
# Join Orders to Geography to get the geo_key

geography_df = spark.table("workspace.amazon_fullfilment_gold.dim_geography")
orders_with_geo = (spark.table("workspace.amazon_fullfilment_silver.orders_silver")
    .join(spark.table("workspace.amazon_fullfilment_silver.customer_silver"), "customer_id", "left")
    .join(spark.table("workspace.amazon_fullfilment_silver.address_silver").alias("address"),  "customer_id", "left")
    .join(spark.table("workspace.amazon_fullfilment_silver.order_items_silver").alias("order_items"), "order_id", "left")
    .join(geography_df, ["City", "Province"], "left") # Join on strings to get the key
)

# Aggregate into the Daily Fact Table
daily_fact_df = (orders_with_geo
    .withColumn("date_key", F.date_format("address.updated_timestamp", "yyyyMMdd").cast("int"))
    .groupBy("date_key", "geo_key")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("order_items.item_count").alias("total_items"), # Assume you have an item count
        F.current_timestamp().alias("_last_updated_timestamp")
    )
)
display(daily_fact_df)

# Save to Gold
#daily_fact_df.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_gold.fact_daily_order_volume")

In [0]:
from pyspark.sql import functions as F

# 1. Pre-aggregate items so we have 1 row per order
items_summarized = (spark.table("workspace.amazon_fullfilment_silver.order_items_silver")
    .groupBy("order_id")
    .agg(F.sum("quantity").alias("total_items_in_order"))
)

# 2. Join orders to Geography and our summarized items
daily_fact_df = (spark.table("workspace.amazon_fullfilment_silver.orders_silver").alias("o")
    .join(spark.table("workspace.amazon_fullfilment_silver.address_silver").alias("a"), "customer_id", "left")
    .join(items_summarized, "order_id", "left")
    .join(geography_df, ["City", "Province"], "left")
    
    # 3. Final Aggregation
    .withColumn("date_key", F.date_format("o.updated_timestamp", "yyyyMMdd").cast("int"))
    .groupBy("date_key", "geo_key")
    .agg(
        F.count("order_id").alias("total_orders"), # Now safe to use count()
        F.sum("total_items_in_order").alias("total_items"),
        F.current_timestamp().alias("_last_updated_timestamp")
    )
)

display(daily_fact_df)

# Save to Gold
daily_fact_df.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_gold.fact_daily_order_volume")

In [0]:
%sql

select total_orders, total_items from workspace.amazon_fullfilment_gold.fact_daily_order_volume
join workspace.amazon_fullfilment_gold.dim_date on dim_date.date_key = fact_daily_order_volume.date_key
join workspace.amazon_fullfilment_gold.dim_geography on dim_geography.geo_key = fact_daily_order_volume.geo_key
where dim_date.date_raw = '2026-05-07' and dim_geography.city = 'New Sheena'